# Post-Training — Advanced
> Section 01 — Topic 06 — advanced

**Prerequisites:** 06-post-training/foundations.ipynb

**What you'll learn:**
- How reward models capture human preferences
- RLHF with PPO — the original InstructGPT approach
- DPO — the simpler alternative that's replacing PPO

**What you'll build:**
- DPO training pipeline from scratch and with TRL

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")

## 1. Reward Modeling — Learning Human Preferences

A **reward model** learns to score model outputs according to human preferences. The training data consists of preference pairs: given a prompt, a human annotator sees two candidate responses and picks the better one. The reward model learns to assign higher scores to preferred responses.

Mathematically, this follows the **Bradley-Terry model** of pairwise comparisons. Given a prompt $x$ and two responses $y_w$ (preferred/"chosen") and $y_l$ (rejected/"lost"), the probability that $y_w$ is preferred is:

$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

where $\sigma$ is the sigmoid function and $r(x, y)$ is the reward model's scalar score. The training loss is:

$$\mathcal{L} = -\log \sigma(r(x, y_w) - r(x, y_l))$$

The architecture is typically a language model with the final unembedding head replaced by a single scalar output — you feed in (prompt + response) and get a single number representing quality.

In [ ]:
# Implement a reward model from scratch

class RewardModel(nn.Module):
    """Reward model: language model backbone + scalar reward head."""
    
    def __init__(self, model_name="gpt2"):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(model_name)
        hidden_size = self.backbone.config.n_embd
        
        # Replace LM head with scalar reward head
        self.reward_head = nn.Linear(hidden_size, 1, bias=False)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.backbone.transformer(
            input_ids=input_ids, 
            attention_mask=attention_mask
        )
        hidden_states = outputs.last_hidden_state
        
        # Use last non-padding token's hidden state as the sequence representation
        if attention_mask is not None:
            # Find position of last real token
            seq_lengths = attention_mask.sum(dim=1) - 1
            last_hidden = hidden_states[range(hidden_states.size(0)), seq_lengths]
        else:
            last_hidden = hidden_states[:, -1]
        
        reward = self.reward_head(last_hidden).squeeze(-1)
        return reward

# Create reward model
reward_model = RewardModel("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Test with preference pair
prompt = "Explain gravity simply."
chosen = "Gravity is the force that pulls objects toward each other. The Earth's gravity keeps us on the ground and causes objects to fall when dropped."
rejected = "Gravity is complicated physics stuff. It's like magnets but not really. Things fall down because of it I guess."

chosen_input = tokenizer(prompt + " " + chosen, return_tensors="pt", padding=True, truncation=True, max_length=128)
rejected_input = tokenizer(prompt + " " + rejected, return_tensors="pt", padding=True, truncation=True, max_length=128)

with torch.no_grad():
    r_chosen = reward_model(**chosen_input)
    r_rejected = reward_model(**rejected_input)

print(f"Reward (chosen):   {r_chosen.item():.4f}")
print(f"Reward (rejected): {r_rejected.item():.4f}")
print(f"\nBefore training, scores are random. Training will teach the model")
print(f"to assign higher rewards to chosen (better) responses.")

In [ ]:
# Train the reward model on preference pairs

preference_data = [
    ("What is Python?",
     "Python is a high-level, interpreted programming language known for its clear syntax and versatility. It's widely used in web development, data science, AI, and automation.",
     "Python is a snake. Also a language I think."),
    ("How do airplanes fly?",
     "Airplanes fly by generating lift through their wing shape. As the plane moves forward, air flows faster over the curved top of the wing than the flat bottom, creating lower pressure above and higher pressure below, which pushes the wing up.",
     "They just do. The engines are really powerful."),
    ("Write hello world in JavaScript.",
     'console.log("Hello, World!"); — This uses the built-in console.log function to print text to the browser console or Node.js terminal.',
     "Just Google it."),
    ("What is 2+2?",
     "2 + 2 = 4.",
     "It depends on your perspective. Numbers are a social construct."),
]

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=1e-5)
losses = []

for epoch in range(10):
    epoch_loss = 0
    for prompt, chosen, rejected in preference_data:
        chosen_enc = tokenizer(prompt + " " + chosen, return_tensors="pt", padding=True, truncation=True, max_length=128)
        rejected_enc = tokenizer(prompt + " " + rejected, return_tensors="pt", padding=True, truncation=True, max_length=128)
        
        r_chosen = reward_model(**chosen_enc)
        r_rejected = reward_model(**rejected_enc)
        
        # Bradley-Terry loss
        loss = -F.logsigmoid(r_chosen - r_rejected).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    losses.append(epoch_loss / len(preference_data))
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}: loss = {losses[-1]:.4f}")

# Verify: trained model should prefer chosen responses
reward_model.eval()
print("\nAfter training — reward scores:")
for prompt, chosen, rejected in preference_data[:2]:
    c_enc = tokenizer(prompt + " " + chosen, return_tensors="pt", truncation=True, max_length=128)
    r_enc = tokenizer(prompt + " " + rejected, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        rc = reward_model(**c_enc).item()
        rr = reward_model(**r_enc).item()
    print(f"  {prompt[:30]}... chosen={rc:.3f} rejected={rr:.3f} ({'✓' if rc > rr else '✗'})")

## 2. RLHF with PPO — The Original InstructGPT Approach

The original alignment approach from InstructGPT (Ouyang et al., 2022) frames the problem as reinforcement learning. The **policy** is the language model, the **action** is generating a token, the **state** is the current sequence, and the **reward** comes from the reward model scoring the complete response.

**Proximal Policy Optimization (PPO)** is used to update the policy because:
1. It prevents the model from changing too much in one update (via clipping)
2. It includes a **KL penalty** that keeps the fine-tuned model close to the original SFT model, preventing reward hacking

The PPO objective for language models is:

$$\mathcal{L}_{PPO} = \mathbb{E}[\min(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t)] - \beta \cdot D_{KL}(\pi_\theta || \pi_{\text{ref}})$$

where $r_t(\theta)$ is the probability ratio between new and old policies, $\hat{A}_t$ is the advantage estimate, and the KL term prevents the model from drifting too far from the reference (SFT) model.

PPO works but is notoriously complex and unstable for LLMs. It requires: a policy model, a reference model, a reward model, and a value model — four models in memory simultaneously.

In [ ]:
# Key PPO components for language models (simplified)

def compute_log_probs(model, input_ids, attention_mask):
    """Compute per-token log probabilities under the model."""
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1]  # Shift: predict next token
    labels = input_ids[:, 1:]         # Shift: actual next tokens
    
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
    return token_log_probs

def compute_kl_divergence(policy_log_probs, reference_log_probs):
    """KL divergence between policy and reference model."""
    return (torch.exp(policy_log_probs) * (policy_log_probs - reference_log_probs)).sum(dim=-1).mean()

def ppo_loss(policy_log_probs, old_log_probs, advantages, clip_epsilon=0.2):
    """Clipped PPO surrogate objective."""
    ratio = torch.exp(policy_log_probs - old_log_probs)
    clipped_ratio = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon)
    
    surr1 = ratio * advantages
    surr2 = clipped_ratio * advantages
    
    return -torch.min(surr1, surr2).mean()

# Demonstrate the clipping mechanism
ratios = torch.linspace(0.5, 1.5, 100)
advantages_pos = torch.ones_like(ratios)  # Positive advantage
advantages_neg = -torch.ones_like(ratios)  # Negative advantage

clip_eps = 0.2
clipped = torch.clamp(ratios, 1 - clip_eps, 1 + clip_eps)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Positive advantage: encourage actions that led to good outcomes
obj_unclipped = ratios * advantages_pos
obj_clipped = torch.min(ratios * advantages_pos, clipped * advantages_pos)
ax1.plot(ratios.numpy(), obj_unclipped.numpy(), 'b--', label='Unclipped', alpha=0.7)
ax1.plot(ratios.numpy(), obj_clipped.numpy(), 'r-', label='Clipped (PPO)', linewidth=2)
ax1.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Policy Ratio π(a)/π_old(a)')
ax1.set_ylabel('Objective')
ax1.set_title('Positive Advantage (good action)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Negative advantage: discourage actions that led to bad outcomes
obj_unclipped_neg = ratios * advantages_neg
obj_clipped_neg = torch.min(ratios * advantages_neg, clipped * advantages_neg)
# For negative advantage, we take max (since we negate in loss)
ax2.plot(ratios.numpy(), obj_unclipped_neg.numpy(), 'b--', label='Unclipped', alpha=0.7)
ax2.plot(ratios.numpy(), torch.max(obj_unclipped_neg, obj_clipped_neg).numpy(), 'r-', label='Clipped (PPO)', linewidth=2)
ax2.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Policy Ratio π(a)/π_old(a)')
ax2.set_ylabel('Objective')
ax2.set_title('Negative Advantage (bad action)')
ax2.legend()
ax2.grid(True, alpha=0.3)

fig.suptitle('PPO Clipping: Prevents Large Policy Updates', y=1.02)
plt.tight_layout()
plt.show()
print("The clip prevents the policy from changing too aggressively in one step.")

## 3. DPO — Direct Preference Optimization

**DPO** (Rafailov et al., 2023) is an elegant breakthrough that eliminates the need for a separate reward model and RL training. The key mathematical insight is that the optimal policy under the RLHF objective has a closed-form relationship with the reward function:

$$r(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

By substituting this into the Bradley-Terry preference model and canceling the partition function $Z(x)$, we get the DPO loss:

$$\mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

This loss directly optimizes the policy using preference data — no reward model, no RL, no value function. You just need:
1. A reference model (the SFT model, frozen)
2. The policy model (starts as a copy of the reference)
3. Preference pairs: (prompt, chosen response, rejected response)

DPO is simpler, more stable, and often matches or exceeds PPO quality. The hyperparameter $\beta$ controls how far the policy can drift from the reference — higher $\beta$ means more conservative updates.

In [ ]:
# Implement DPO loss from scratch

def dpo_loss(policy_chosen_logps, policy_rejected_logps, 
             reference_chosen_logps, reference_rejected_logps, 
             beta=0.1):
    """
    Compute DPO loss.
    
    Args:
        policy_chosen_logps: Log probs of chosen responses under policy
        policy_rejected_logps: Log probs of rejected responses under policy
        reference_chosen_logps: Log probs of chosen under reference (frozen)
        reference_rejected_logps: Log probs of rejected under reference (frozen)
        beta: Temperature parameter (higher = more conservative)
    
    Returns:
        loss: DPO loss
        chosen_rewards: Implicit rewards for chosen
        rejected_rewards: Implicit rewards for rejected
    """
    # Log ratios: how much does the policy differ from reference?
    chosen_log_ratios = policy_chosen_logps - reference_chosen_logps
    rejected_log_ratios = policy_rejected_logps - reference_rejected_logps
    
    # DPO logits: the model's implicit preference
    logits = beta * (chosen_log_ratios - rejected_log_ratios)
    
    # Binary cross-entropy: maximize probability of preferring chosen
    loss = -F.logsigmoid(logits).mean()
    
    # Implicit rewards (for monitoring)
    chosen_rewards = beta * chosen_log_ratios.detach()
    rejected_rewards = beta * rejected_log_ratios.detach()
    
    return loss, chosen_rewards, rejected_rewards

# Demo: show how DPO loss behaves
# When policy agrees with preferences (chosen has higher log prob ratio)
good_case = dpo_loss(
    policy_chosen_logps=torch.tensor([-1.0]),
    policy_rejected_logps=torch.tensor([-3.0]),
    reference_chosen_logps=torch.tensor([-1.5]),
    reference_rejected_logps=torch.tensor([-1.5]),
    beta=0.1
)

# When policy disagrees (rejected has higher log prob ratio)
bad_case = dpo_loss(
    policy_chosen_logps=torch.tensor([-3.0]),
    policy_rejected_logps=torch.tensor([-1.0]),
    reference_chosen_logps=torch.tensor([-1.5]),
    reference_rejected_logps=torch.tensor([-1.5]),
    beta=0.1
)

print(f"Policy prefers chosen (correct):  loss = {good_case[0].item():.4f}")
print(f"Policy prefers rejected (wrong):  loss = {bad_case[0].item():.4f}")
print(f"\nHigher loss when policy disagrees with human preferences → gradient pushes policy to agree.")

In [ ]:
# Visualize DPO loss landscape
betas = [0.05, 0.1, 0.2, 0.5]
log_ratio_diffs = torch.linspace(-3, 3, 100)  # chosen_ratio - rejected_ratio

fig, ax = plt.subplots(figsize=(10, 6))
for beta in betas:
    losses = -F.logsigmoid(beta * log_ratio_diffs)
    ax.plot(log_ratio_diffs.numpy(), losses.numpy(), label=f'β = {beta}', linewidth=2)

ax.axvline(x=0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Log-ratio difference (chosen - rejected)')
ax.set_ylabel('DPO Loss')
ax.set_title('DPO Loss vs Policy Preference Strength')
ax.annotate('Policy prefers\nrejected', xy=(-2, 0.5), fontsize=10, color='red')
ax.annotate('Policy prefers\nchosen', xy=(1.5, 0.5), fontsize=10, color='green')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Higher β = more conservative (steeper loss, stays closer to reference model)")

## 4. DPO in Practice — Using TRL

TRL's `DPOTrainer` handles the full DPO pipeline: computing log probabilities for both policy and reference models, managing the reference model (frozen), computing the DPO loss, and training with standard optimizers.

The dataset format is straightforward: each example has a `prompt`, a `chosen` response, and a `rejected` response. The trainer automatically handles tokenization, padding, and log probability computation.

In [ ]:
# DPO with TRL — production approach
try:
    from trl import DPOConfig, DPOTrainer
    from datasets import Dataset
    
    # Prepare preference dataset
    dpo_data = []
    for prompt, chosen, rejected in preference_data:
        dpo_data.append({
            "prompt": prompt,
            "chosen": chosen,
            "rejected": rejected,
        })
    
    dataset = Dataset.from_list(dpo_data)
    print(f"DPO Dataset: {len(dataset)} preference pairs")
    print(f"\nExample:")
    print(f"  Prompt:   {dataset[0]['prompt']}")
    print(f"  Chosen:   {dataset[0]['chosen'][:80]}...")
    print(f"  Rejected: {dataset[0]['rejected'][:80]}...")
    
    # DPO config
    dpo_config = DPOConfig(
        output_dir="./dpo_output",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        learning_rate=5e-6,  # Lower LR than SFT
        beta=0.1,            # DPO temperature
        logging_steps=1,
        save_strategy="no",
        max_length=256,
        max_prompt_length=128,
    )
    
    print("\nDPOConfig created. In production:")
    print("  trainer = DPOTrainer(")
    print("      model=sft_model,")
    print("      ref_model=None,  # auto-creates from model")
    print("      args=dpo_config,")
    print("      train_dataset=dataset,")
    print("      tokenizer=tokenizer,")
    print("  )")
    print("  trainer.train()")
    
except ImportError:
    print("Install TRL: pip install trl")
    print("\nDPOTrainer handles:")
    print("  - Reference model management (frozen copy)")
    print("  - Log probability computation for both models")
    print("  - DPO loss with configurable beta")
    print("  - Reward accuracy logging (for monitoring)")

## 5. Comparing RLHF vs DPO

Both approaches optimize the same underlying objective — aligning model outputs with human preferences — but they take very different paths to get there.

In [ ]:
# Comparison table
comparison = {
    "Aspect": ["Models needed", "Training stability", "Implementation complexity", 
               "Memory requirement", "Hyperparameter sensitivity", "Quality (empirical)",
               "Training speed", "When to use"],
    "RLHF (PPO)": [
        "4 (policy, reference, reward, value)",
        "Often unstable, requires careful tuning",
        "Very complex — RL loop, GAE, clipping",
        "4x model size in memory",
        "High — many hyperparams to tune",
        "Good, sometimes better for diverse rewards",
        "Slow — generation + RL updates per step",
        "Diverse reward signals, online learning",
    ],
    "DPO": [
        "2 (policy, reference)",
        "Very stable — just supervised learning",
        "Simple — standard training loop",
        "2x model size in memory",
        "Low — mainly just beta",
        "Good, matches PPO on most benchmarks",
        "Fast — no generation during training",
        "Offline preference data, simplicity needed",
    ],
}

print(f"{'Aspect':<30} {'RLHF (PPO)':<45} {'DPO':<40}")
print("=" * 115)
for i, aspect in enumerate(comparison["Aspect"]):
    print(f"{aspect:<30} {comparison['RLHF (PPO)'][i]:<45} {comparison['DPO'][i]:<40}")

print("\n" + "=" * 115)
print("\nVerdict: DPO has largely replaced PPO for most use cases.")
print("PPO is still used when: online/iterative RLHF is needed, reward is not pairwise,")
print("or you need to optimize multiple reward signals simultaneously.")

In [ ]:
# Visualize the training pipeline comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# RLHF Pipeline
steps_rlhf = ['Base\nModel', 'SFT\nModel', 'Train\nReward\nModel', 'PPO\nTraining', 'Aligned\nModel']
colors_rlhf = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
ax1.barh(range(len(steps_rlhf)), [1]*len(steps_rlhf), color=colors_rlhf, height=0.6)
for i, step in enumerate(steps_rlhf):
    ax1.text(0.5, i, step, ha='center', va='center', fontsize=10, fontweight='bold')
ax1.set_xlim(0, 1)
ax1.set_title('RLHF Pipeline (complex)', fontsize=14)
ax1.set_yticks([])
ax1.set_xticks([])
for i in range(len(steps_rlhf)-1):
    ax1.annotate('→', xy=(0.95, i+0.3), fontsize=16)

# DPO Pipeline
steps_dpo = ['Base\nModel', 'SFT\nModel', 'DPO\nTraining', 'Aligned\nModel']
colors_dpo = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
ax2.barh(range(len(steps_dpo)), [1]*len(steps_dpo), color=colors_dpo, height=0.6)
for i, step in enumerate(steps_dpo):
    ax2.text(0.5, i, step, ha='center', va='center', fontsize=10, fontweight='bold')
ax2.set_xlim(0, 1)
ax2.set_title('DPO Pipeline (simpler)', fontsize=14)
ax2.set_yticks([])
ax2.set_xticks([])
for i in range(len(steps_dpo)-1):
    ax2.annotate('→', xy=(0.95, i+0.3), fontsize=16)

plt.tight_layout()
plt.show()

## Summary

**Key takeaways:**
- **Reward models** learn human preferences using Bradley-Terry pairwise comparison loss
- **RLHF/PPO** frames alignment as RL: policy (LM) optimized against reward model with KL constraint. Complex but powerful
- **DPO** collapses reward modeling and RL into a single supervised loss. Simpler, more stable, similar quality
- The DPO loss is: $-\log\sigma(\beta(\log\frac{\pi_\theta(y_w)}{\pi_{ref}(y_w)} - \log\frac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}))$
- $\beta$ controls conservatism: higher $\beta$ = stay closer to reference model
- DPO has largely replaced PPO in practice; PPO still used for online/iterative alignment

**Next:** [06-post-training/expert.ipynb](expert.ipynb) — Constitutional AI, ORPO, KTO, process reward models